# Fine-Tuning Setup
Run this notebook once. Preprocesses all 148 subjects and saves to disk so other fine-tuning notebooks load instantly.

In [1]:
import os, sys, json
import numpy as np
import torch
from tqdm import tqdm
from sklearn.model_selection import train_test_split

sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/BrainOmni')
sys.path.insert(0, '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project')
from brainomni.model import BrainOmni
from src.preprocessing import preprocess_subject, FS_TARGET, SEGMENT_SECS

device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print('Device:', device)

W0708 11:31:26.174000 28676 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Device: mps


In [2]:
CKPT_PATH = '/Users/faenegoro/Documents/Neuroimaging/BrainOmni/ckpt_collection/tiny'
SAVE_DIR  = '../data/preprocessed'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(os.path.join(CKPT_PATH, 'model_cfg.json')) as f:
    model_config = json.load(f)
base_checkpoint = torch.load(
    os.path.join(CKPT_PATH, 'BrainOmni.pt'), map_location='cpu', weights_only=True
)

model = BrainOmni(**model_config)
model.load_state_dict(base_checkpoint, strict=False)
model = model.to(device).eval()
print('Model loaded.')

Model loaded.


/opt/miniconda3/envs/brainomni/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [3]:
DATA_DIR = '/Users/faenegoro/Documents/Neuroimaging/meg_ad_project/data/meg_mri'
files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('_meg_rest_60sec.mat')])

labels_dict = {}
for f in files:
    sid = f.split('_')[0]
    labels_dict[sid] = 0 if int(sid[4:]) <= 70 else 1

subject_ids = [f.split('_')[0] for f in files]
labels = np.array([labels_dict[sid] for sid in subject_ids])
print(f'Subjects: {len(labels)} | Controls: {(labels==0).sum()} | AD: {(labels==1).sum()}')

Subjects: 148 | Controls: 70 | AD: 78


In [4]:
SEGMENT_LEN = SEGMENT_SECS * FS_TARGET  # 7680

all_seg1, all_seg2, all_pos, all_st = [], [], [], []

for filename in tqdm(files, desc='Preprocessing'):
    mat_path = os.path.join(DATA_DIR, filename)
    signal, pos, sensor_type = preprocess_subject(mat_path)
    all_seg1.append(signal[:, :SEGMENT_LEN])
    all_seg2.append(signal[:, SEGMENT_LEN:SEGMENT_LEN*2])
    all_pos.append(pos)
    all_st.append(sensor_type)

print(f'Preprocessed {len(all_seg1)} subjects | Segment shape: {all_seg1[0].shape}')

Preprocessing: 100%|██████████| 148/148 [00:49<00:00,  3.01it/s]

Preprocessed 148 subjects | Segment shape: (271, 7680)


In [5]:
idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, stratify=labels, random_state=123)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, stratify=labels[temp_idx], random_state=123)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

Train: 118 | Val: 15 | Test: 15


In [6]:
# Tokenize all 148 subjects — no filtering needed.
# BrainOmni tokenizer uses Perceiver cross-attention (16 learnable queries)
# so output is always (16, NT, 256) regardless of input channel count.
proj_out1, proj_out2 = [], []
failed = []

with torch.no_grad():
    for i in tqdm(range(len(all_seg1)), desc='Tokenizing'):
        try:
            s1    = torch.tensor(all_seg1[i]).unsqueeze(0).float().to(device)
            s2    = torch.tensor(all_seg2[i]).unsqueeze(0).float().to(device)
            pos_t = torch.tensor(all_pos[i]).unsqueeze(0).float().to(device)
            st_t  = torch.tensor(all_st[i]).unsqueeze(0).long().to(device)

            x1, _ = model.tokenizer.tokenize(s1, pos_t, st_t, model.overlap_ratio)
            x2, _ = model.tokenizer.tokenize(s2, pos_t, st_t, model.overlap_ratio)
            B, C, W, D = x1.shape
            neuro = model.tokenizer.encoder.neuros.type_as(x1).detach().view(1, C, 1, -1)
            x1 = model.projection(x1 + neuro)
            x2 = model.projection(x2 + neuro)

            proj_out1.append(x1.squeeze(0).cpu().numpy())
            proj_out2.append(x2.squeeze(0).cpu().numpy())
        except Exception as e:
            print(f'  Subject {i} failed: {e}')
            failed.append(i)

keep         = [i for i in range(len(all_seg1)) if i not in failed]
labels_save  = labels[keep]
proj_out1    = np.array(proj_out1, dtype=np.float32)
proj_out2    = np.array(proj_out2, dtype=np.float32)
print(f'Tokenized {len(proj_out1)}/{len(all_seg1)} subjects | Shape: {proj_out1.shape}')
print(f'Controls: {(labels_save==0).sum()} | AD: {(labels_save==1).sum()}')

# Create 80/10/10 split
idx_s = np.arange(len(labels_save))
train_idx, temp_idx = train_test_split(idx_s, test_size=0.2, stratify=labels_save, random_state=123)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, stratify=labels_save[temp_idx], random_state=123)
print(f'Split | Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

np.save(os.path.join(SAVE_DIR, 'proj_out1.npy'), proj_out1)
np.save(os.path.join(SAVE_DIR, 'proj_out2.npy'), proj_out2)
np.save(os.path.join(SAVE_DIR, 'labels.npy'), labels_save)
np.savez(os.path.join(SAVE_DIR, 'split.npz'),
         train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

print('\nSaved to', SAVE_DIR)
print('Files:', [f for f in os.listdir(SAVE_DIR) if not f.endswith('.pt')])

Tokenizing: 100%|██████████| 148/148 [01:05<00:00,  2.24it/s]


Tokenized 148/148 subjects | Shape: (148, 16, 160, 256)
Controls: 70 | AD: 78
Split | Train: 118 | Val: 15 | Test: 15

Saved to ../data/preprocessed
Files: ['labels.npy', 'proj_out2.npy', 'proj_out1.npy', 'all_seg2.npy', 'all_seg1.npy', 'split.npz', 'all_st.npy', 'all_pos.npy']
